# Corpus Dialectal Latinoamericano — Construccion
**Paper:** Dialectal Robustness in Spanish-to-English Speech Translation  
**Autores:** Berly Diaz-Castro, Roxana Flores-Quispe (UNSA, Arequipa, Peru)

Descarga 6 datasets dialectales latinoamericanos y genera referencias en ingles con Llama 3.3 70B (Groq).

| Dialecto | Fuente | Codigo |
|----------|--------|--------|
| Peruano | OpenSLR 73 | `pe` |
| Puertorriqueno | OpenSLR 74 | `pr` |
| Venezolano | OpenSLR 75 | `ve` |
| Colombiano | HuggingFace ylacombe | `co` |
| Chileno | HuggingFace ylacombe | `cl` |
| Argentino | HuggingFace ylacombe | `ar` |

In [1]:
!pip install -q groq datasets pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive, userdata
from datasets import load_dataset, concatenate_datasets
from groq import Groq
import numpy as np
import pandas as pd
import urllib.request
import os
import time
import warnings
warnings.filterwarnings('ignore')

drive.mount('/content/drive', force_remount=True)
BASE_DIR = '/content/drive/MyDrive/MSR_Dialectal'
os.makedirs(f'{BASE_DIR}/tsv', exist_ok=True)
os.makedirs(f'{BASE_DIR}/csv', exist_ok=True)

# Agrega tu clave en Colab: icono de llave (Secrets) → GROQ_API_KEY
GROQ_API_KEY        = userdata.get('GROQ_API_KEY')
SAMPLES_PER_DIALECT = 500
RANDOM_STATE        = 42

client = Groq(api_key=GROQ_API_KEY)

test = client.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=[
        {'role': 'system', 'content': 'Translate Spanish to English. Output only the translation.'},
        {'role': 'user',   'content': 'Buenos dias, como esta usted?'}
    ],
    temperature=0.1, max_tokens=64
)
print('Groq OK:', test.choices[0].message.content)
print(f'BASE_DIR: {BASE_DIR}')

## 1. Descarga TSV de OpenSLR (Peru, Puerto Rico, Venezuela)

In [3]:
OPENSLR_TSV = {
    'pe_female': 'https://www.openslr.org/resources/73/line_index_female.tsv',
    'pe_male':   'https://www.openslr.org/resources/73/line_index_male.tsv',
    'pr_female': 'https://www.openslr.org/resources/74/line_index_female.tsv',
    've_female': 'https://www.openslr.org/resources/75/line_index_female.tsv',
    've_male':   'https://www.openslr.org/resources/75/line_index_male.tsv',
}

for name, url in OPENSLR_TSV.items():
    out = f'{BASE_DIR}/tsv/{name}.tsv'
    if not os.path.exists(out):
        print(f'Descargando {name}...')
        urllib.request.urlretrieve(url, out)
    print(f'  ok {out}')

print('\nTSV descargados en Drive.')

  ok /content/drive/MyDrive/MSR_Dialectal/tsv/pe_female.tsv
  ok /content/drive/MyDrive/MSR_Dialectal/tsv/pe_male.tsv
  ok /content/drive/MyDrive/MSR_Dialectal/tsv/pr_female.tsv
  ok /content/drive/MyDrive/MSR_Dialectal/tsv/ve_female.tsv
  ok /content/drive/MyDrive/MSR_Dialectal/tsv/ve_male.tsv

TSV descargados en Drive.


## 2. Carga HuggingFace (Colombia, Chile, Argentina)

In [4]:
HF_DATASETS = {
    'co': 'ylacombe/google-colombian-spanish',
    'cl': 'ylacombe/google-chilean-spanish',
    'ar': 'ylacombe/google-argentinian-spanish',
}

hf_data = {}
for dialect, hf_id in HF_DATASETS.items():
    print(f'Cargando {dialect} ({hf_id})...')
    parts = []
    for config in ['female', 'male']:
        try:
            ds = load_dataset(hf_id, config, split='train')
            parts.append(ds)
            print(f'  {config}: {len(ds)} muestras')
        except Exception:
            print(f'  {config}: no disponible')
    if parts:
        hf_data[dialect] = concatenate_datasets(parts)
        print(f'  => total {len(hf_data[dialect])} | cols: {hf_data[dialect].column_names}')

print('\nDatasets HF listos.')

Cargando co (ylacombe/google-colombian-spanish)...


README.md:   0%|          | 0.00/907 [00:00<?, ?B/s]

female/train-00000-of-00003-f5351e75d615(…):   0%|          | 0.00/343M [00:00<?, ?B/s]

female/train-00001-of-00003-61a1576d9736(…):   0%|          | 0.00/343M [00:00<?, ?B/s]

female/train-00002-of-00003-3c68c96f13d2(…):   0%|          | 0.00/353M [00:00<?, ?B/s]

...

  female: 3921 muestras


male/train-00000-of-00002-920b805572ae22(…):   0%|          | 0.00/357M [00:00<?, ?B/s]

male/train-00001-of-00002-f6f0bfbdc6bb1d(…):   0%|          | 0.00/350M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1818 [00:00<?, ? examples/s]

  male: 1818 muestras
  => total 5739 | cols: ['audio', 'text', 'speaker_id']

Datasets HF listos.


## 3. Inventario de muestras

In [5]:
openslr_map = {
    'pe': ('Peru',        'OpenSLR 73', ['pe_female', 'pe_male']),
    'pr': ('Puerto Rico', 'OpenSLR 74', ['pr_female']),
    've': ('Venezuela',   'OpenSLR 75', ['ve_female', 've_male']),
}
hf_map = {'co': 'Colombia', 'cl': 'Chile', 'ar': 'Argentina'}

inventory = []
for dialect, (country, source, files) in openslr_map.items():
    total = sum(
        len(pd.read_csv(f'{BASE_DIR}/tsv/{f}.tsv', sep='\t', header=None))
        for f in files if os.path.exists(f'{BASE_DIR}/tsv/{f}.tsv')
    )
    inventory.append({'dialecto': country, 'codigo': dialect, 'fuente': source, 'muestras': total})

for dialect, ds in hf_data.items():
    inventory.append({'dialecto': hf_map[dialect], 'codigo': dialect, 'fuente': 'HuggingFace', 'muestras': len(ds)})

df_inv = pd.DataFrame(inventory)
df_inv['para_evaluacion'] = df_inv['muestras'].apply(lambda x: min(x, SAMPLES_PER_DIALECT))
print(df_inv.to_string(index=False))
print(f'\nTotal disponible:      {df_inv["muestras"].sum():,}')
print(f'Total para evaluacion: {df_inv["para_evaluacion"].sum():,}')

   dialecto codigo      fuente  muestras  para_evaluacion
       Peru     pe  OpenSLR 73      5447              500
Puerto Rico     pr  OpenSLR 74       617              500
  Venezuela     ve  OpenSLR 75      3357              500
   Colombia     co HuggingFace      4903              500
      Chile     cl HuggingFace      4374              500
  Argentina     ar HuggingFace      5739              500

Total disponible:      24,437
Total para evaluacion: 3,000


## 4. Construccion del DataFrame de textos

In [6]:
rng = np.random.default_rng(RANDOM_STATE)
all_texts = []

# OpenSLR
for dialect, (country, source, files) in openslr_map.items():
    rows = []
    for f in files:
        path = f'{BASE_DIR}/tsv/{f}.tsv'
        if os.path.exists(path):
            gender = f.split('_')[1]
            df = pd.read_csv(path, sep='\t', header=None, names=['file_id', 'transcription'])
            df['dialect'] = dialect
            df['gender']  = gender
            df['source']  = f'openslr_{dialect}'
            rows.append(df)
    if rows:
        combined = pd.concat(rows).dropna(subset=['transcription']).reset_index(drop=True)
        n = min(SAMPLES_PER_DIALECT, len(combined))
        idx = sorted(rng.choice(len(combined), size=n, replace=False))
        all_texts.append(combined.iloc[idx][['file_id', 'transcription', 'dialect', 'gender', 'source']])

# HuggingFace
for dialect, ds in hf_data.items():
    df = ds.to_pandas()
    text_col = next((c for c in ['text', 'sentence', 'transcription'] if c in df.columns), df.columns[1])
    df = df[[text_col]].rename(columns={text_col: 'transcription'})
    df['file_id'] = [f'{dialect}_{i}' for i in range(len(df))]
    df['dialect'] = dialect
    df['gender']  = 'mixed'
    df['source']  = f'hf_{dialect}'
    n = min(SAMPLES_PER_DIALECT, len(df))
    idx = sorted(rng.choice(len(df), size=n, replace=False))
    all_texts.append(df.iloc[idx][['file_id', 'transcription', 'dialect', 'gender', 'source']])

corpus_es = pd.concat(all_texts).reset_index(drop=True)
print(f'Total: {len(corpus_es)} muestras')
print(corpus_es.groupby('dialect')['transcription'].count().rename('muestras'))

out_es = f'{BASE_DIR}/csv/corpus_es_dialectal.csv'
corpus_es.to_csv(out_es, index=False)
print(f'\nGuardado: {out_es}')
corpus_es.head(3)

Total: 3000 muestras
dialect
ar    500
cl    500
co    500
pe    500
pr    500
ve    500
Name: muestras, dtype: int64

Guardado: /content/drive/MyDrive/MSR_Dialectal/csv/corpus_es_dialectal.csv


,file_id,transcription,dialect,gender,source
0,pef_02436_01114402488,¿Qué te parece los más grandes éxitos en español?,pe,female,openslr_pe
1,pef_02121_01832964757,¿Quiero saber la información nutricional que t...,pe,female,openslr_pe
2,pef_00610_01085094011,Los mejores lugares para entablar primeras cit...,pe,female,openslr_pe


## 5. Traduccion al ingles (Llama 3.3 70B via Groq)

In [9]:
import torch
from transformers import MarianMTModel, MarianTokenizer
from tqdm import tqdm

model_name = "Helsinki-NLP/opus-mt-es-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name).to("cuda")
model.eval()
print("Modelo Helsinki-NLP/opus-mt-es-en cargado en GPU.")

def translate_all(texts, batch_size=64):
    results = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Traduciendo"):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")
        with torch.no_grad():
            outputs = model.generate(**inputs)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        results.extend(decoded)
    return results

# ── Retoma desde el CSV guardado si existe ───────────────────────────────────
out_full = f'{BASE_DIR}/csv/corpus_dialectal_completo.csv'

if os.path.exists(out_full):
    print(f'Cargando CSV parcial: {out_full}')
    corpus_es = pd.read_csv(out_full)
else:
    corpus_es['reference_en'] = ''

mask = corpus_es['reference_en'].fillna('').str.strip().eq('')
todo_idx = corpus_es.index[mask].tolist()
print(f'Ya traducidas: {(~mask).sum()} | Pendientes: {len(todo_idx)}')

if todo_idx:
    texts = corpus_es.loc[todo_idx, 'transcription'].tolist()
    translations = translate_all(texts)
    for idx, t in zip(todo_idx, translations):
        corpus_es.at[idx, 'reference_en'] = t

corpus_es.to_csv(out_full, index=False)
print(f'\nGuardado: {out_full}')
corpus_es[['dialect', 'transcription', 'reference_en']].head(5)

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.59M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Modelo Helsinki-NLP/opus-mt-es-en cargado en GPU.
Cargando CSV parcial: /content/drive/MyDrive/MSR_Dialectal/csv/corpus_dialectal_completo.csv
Ya traducidas: 1007 | Pendientes: 1993


Traduciendo: 100%|██████████| 32/32 [00:25<00:00,  1.26it/s]


Guardado: /content/drive/MyDrive/MSR_Dialectal/csv/corpus_dialectal_completo.csv


,dialect,transcription,reference_en
0,pe,¿Qué te parece los más grandes éxitos en español?,What do you think of the greatest hits in Span...
1,pe,¿Quiero saber la información nutricional que t...,I want to know the nutritional information of ...
2,pe,Los mejores lugares para entablar primeras cit...,The best places for first dates as a couple ar...
3,pe,Se armó una polémica por el asunto de que Mara...,A controversy arose over the issue of Maradona...
4,pe,¿Te gusta el vino?,Do you like wine?


## 6. Validacion y resumen final

In [ ]:
empty = corpus_es['reference_en'].str.strip().eq('')
short = corpus_es['reference_en'].str.split().str.len() < 2
print(f'Vacias: {empty.sum()} | Cortas: {short.sum()} | Validas: {(~empty & ~short).sum()}/{len(corpus_es)}')

print('\n=== Muestra para revision manual ===')
sample_idx = rng.choice(len(corpus_es), size=6, replace=False)
for _, row in corpus_es.iloc[sample_idx].iterrows():
    print(f"[{row['dialect']}] ES: {row['transcription']}")
    print(f"         EN: {row['reference_en']}\n")

pais = {'pe':'Peru','pr':'Puerto Rico','ve':'Venezuela','co':'Colombia','cl':'Chile','ar':'Argentina'}
stats = corpus_es.groupby('dialect').agg(
    muestras     = ('transcription', 'count'),
    palabras_es  = ('transcription', lambda x: round(x.str.split().str.len().mean(), 1)),
    palabras_en  = ('reference_en',  lambda x: round(x.str.split().str.len().mean(), 1))
)
stats.index = stats.index.map(pais)
print('\n=== Resumen ===')
print(stats.to_string())
print(f'\nTotal: {len(corpus_es)} pares | Referencia: Llama 3.3 70B (Groq)')
print(f'Archivo final: {out_full}')

Vacias: 0 | Cortas: 0 | Validas: 3000/3000

=== Muestra para revision manual ===
[pe] ES: En su teléfono hay una pestaña que dice aplicaciones, apriete el icono de color azul
         EN: On your phone, there is a tab that says applications, press the blue icon.

[pe] ES: Hay un refrán que dice, todo lo que sube baja pero no tan fácilmente
         EN: There's a saying that goes, everything that goes up comes down, but not so easily.

[pr] ES: Me gusta mucho como actúa ese actor es el actor que más me hace reír
         EN: I really like how that actor performs, he's the one who makes me laugh the most.

[ar] ES: Estoy muy emocionada porque me voy a Italia
         EN: I'm so excited because I'm going to Italy.

[cl] ES: La realidad es bastante ambigua
         EN: The reality is quite ambiguous.

[ar] ES: ¿Podés verificar si hay algún café cerca?
         EN: Can you check to see if there's any coffee nearby?


=== Resumen ===
             muestras  palabras_es  palabras_en
dialect   

: 